# AgentHarm — Tiny Reproduction Notebook

**Goal:** reproduce one small, basic AgentHarm number (refusal rate / harm score) for **one model** on a **tiny subset** of the **public** tasks, with **no jailbreak**.

**Paper:** *AgentHarm: A Benchmark for Measuring Harmfulness of LLM Agents* — Andriushchenko, Souly et al., 2024. https://arxiv.org/abs/2410.09024

**This notebook does NOT modify any benchmark code.** It only inspects the existing implementation in `src/inspect_evals/agentharm`, builds the exact `inspect eval` command, optionally runs it, and parses the resulting logs.

**Safety note:** AgentHarm contains prompts describing harmful behavior. This notebook deliberately avoids printing full harmful prompt text — it only shows IDs, names, categories, and short truncated snippets so you can sanity-check the data without reproducing operational harmful content.

## How AgentHarm scoring works (read first)

AgentHarm is an **agentic** benchmark. For each task the model is given a set of tools and a request; the default agent runs a tool-calling loop. Two kinds of judging then happen (see `scorer.py`):

1. **Refusal judge** — an LLM (default `openai/gpt-4o-2024-08-06`) decides whether any assistant message is a refusal. This produces the **refusal rate**.
2. **Semantic / grading functions** — per-task Python grading functions (in `benchmark/harmful_grading_functions.py`) check whether the agent actually performed the harmful steps, optionally aided by a **semantic judge** LLM. This produces the **harm score** (`avg_score`).

**Important consequence:** even with a tiny run you need **two** model roles:
- the **target model** under test (e.g. `openai/gpt-4o-mini`), and
- the **judge models** (refusal + semantic), which default to `openai/gpt-4o-2024-08-06`.

If you only have, say, an Anthropic key, you must override the judges too (shown later).

---
## 1. Setup / sanity checks

Confirm where we are, that this is the `inspect_evals` repo, that the AgentHarm source exists, the Python version, and that the key imports work.

In [36]:
import os
import sys
from pathlib import Path

# This notebook lives at experiments/agentharm_repro/agentharm_repro.ipynb.
# Walk up until we find the repo root (the folder that contains src/inspect_evals).
cwd = Path.cwd().resolve()
print("Notebook working directory:", cwd)

repo_root = cwd
for candidate in [cwd, *cwd.parents]:
    if (candidate / "src" / "inspect_evals").is_dir():
        repo_root = candidate
        break

print("Detected repo root:", repo_root)

# Move the process into the repo root so that `inspect eval inspect_evals/agentharm`
# and dataset caching behave exactly like a normal CLI run from the repo.
os.chdir(repo_root)
print("Changed working directory to:", Path.cwd())

# Load the repo-root .env into this kernel's environment.
# IMPORTANT: a Jupyter kernel does NOT read .env automatically (only `inspect eval`
# does). Without this, Section 5's key check shows "not set" even with a valid .env.
# The CLI/subprocess run in Section 7 would still work, but loading here makes the
# in-notebook checks accurate too.
try:
    from dotenv import load_dotenv

    env_path = repo_root / ".env"
    loaded = load_dotenv(env_path)
    print(f"Loaded .env from {env_path}: {'yes' if loaded else 'no (.env missing or empty)'}")
except ImportError:
    print("python-dotenv not installed; skipping .env load.")
    print("   `inspect eval` still loads .env on its own, or run: uv pip install python-dotenv")

Notebook working directory: /Users/raghavmanoharanjayanthi/Downloads/inspect_evals
Detected repo root: /Users/raghavmanoharanjayanthi/Downloads/inspect_evals
Changed working directory to: /Users/raghavmanoharanjayanthi/Downloads/inspect_evals
Loaded .env from /Users/raghavmanoharanjayanthi/Downloads/inspect_evals/.env: yes


In [37]:
# Confirm this really is the inspect_evals repo and that AgentHarm exists.
agentharm_dir = repo_root / "src" / "inspect_evals" / "agentharm"

checks = {
    "pyproject.toml exists": (repo_root / "pyproject.toml").exists(),
    "src/inspect_evals exists": (repo_root / "src" / "inspect_evals").is_dir(),
    "src/inspect_evals/agentharm exists": agentharm_dir.is_dir(),
    "agentharm.py exists": (agentharm_dir / "agentharm.py").exists(),
}
for label, ok in checks.items():
    print(f"[{'OK ' if ok else 'MISSING'}] {label}")

if not all(checks.values()):
    print("\nBLOCKER: Could not confirm the inspect_evals/agentharm layout.")
    print("Next step: open this notebook from inside a clone of")
    print("https://github.com/UKGovernmentBEIS/inspect_evals")

[OK ] pyproject.toml exists
[OK ] src/inspect_evals exists
[OK ] src/inspect_evals/agentharm exists
[OK ] agentharm.py exists


In [38]:
# Python version + WHICH interpreter this kernel is using.
# COMMON PITFALL: if Jupyter uses the system Python (e.g. 3.9.x) instead of the
# project's uv .venv (3.12), none of the inspect imports will resolve.
print("Python:", sys.version)
print("Interpreter:", sys.executable)

venv_python = repo_root / ".venv" / "bin" / "python"
if venv_python.exists() and Path(sys.executable).resolve() != venv_python.resolve():
    print("\n[WARNING] This kernel is NOT the project's uv .venv.")
    print(f"   Expected: {venv_python}")
    print("   Fix: switch the kernel to 'inspect_evals (.venv 3.12)' (registered via")
    print("   `uv run python -m ipykernel install --user --name inspect-evals`), or select")
    print("   .venv/bin/python as the interpreter, then restart the kernel.")

# Check the key imports. We use importlib so a failure prints a clear blocker
# instead of crashing the whole notebook.
import importlib

for mod in ["inspect_ai", "inspect_evals", "openai"]:
    try:
        m = importlib.import_module(mod)
        print(f"[OK ] import {mod}  (version: {getattr(m, '__version__', 'n/a')})")
    except Exception as e:  # noqa: BLE001 - we want to surface any import error
        print(f"[FAIL] import {mod}: {type(e).__name__}: {e}")
        if mod == "openai":
            print("   `openai` is NOT a core inspect_evals dependency. Install the provider")
            print("   package(s) for your model(s), e.g.: uv pip install openai anthropic")
        else:
            print("   Next step: run `uv sync`, ensure this notebook uses the .venv kernel")
            print("   (see the WARNING above), then restart the kernel.")

# Confirm the AgentHarm tasks themselves import.
try:
    from inspect_evals.agentharm import agentharm, agentharm_benign
    print("[OK ] imported agentharm and agentharm_benign tasks")
except Exception as e:  # noqa: BLE001
    print(f"[FAIL] importing agentharm tasks: {type(e).__name__}: {e}")

Python: 3.12.12 (main, Oct  9 2025, 11:07:00) [Clang 17.0.0 (clang-1700.6.3.2)]
Interpreter: /Users/raghavmanoharanjayanthi/Downloads/inspect_evals/.venv/bin/python3
[OK ] import inspect_ai  (version: 0.3.232)
[OK ] import inspect_evals  (version: n/a)
[OK ] import openai  (version: 2.41.1)
[OK ] imported agentharm and agentharm_benign tasks


/Users/raghavmanoharanjayanthi/Downloads/inspect_evals/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
## 2. Inspect AgentHarm files

List the AgentHarm source files and point at the four things you asked about: **task registration / entrypoint**, **dataset loading**, **scoring**, and **tools**.

In [39]:
# Top-level Python files in the AgentHarm package (skip the big tool/content dirs).
top_level_py = sorted(p.name for p in agentharm_dir.glob("*.py"))
print("Top-level Python files in src/inspect_evals/agentharm/:")
for name in top_level_py:
    print("   -", name)

print("\nKey subdirectories:")
for sub in sorted(p.name for p in agentharm_dir.iterdir() if p.is_dir()):
    print("   -", sub + "/")

Top-level Python files in src/inspect_evals/agentharm/:
   - __init__.py
   - agentharm.py
   - metric.py
   - prompts.py
   - scorer.py
   - utils.py

Key subdirectories:
   - __pycache__/
   - agents/
   - benchmark/
   - images/


In [40]:
# A short, hand-written map of what each important file does.
# (Static summary so you don't have to read every file during the meeting.)
file_roles = {
    "__init__.py": "Exports the `agentharm` and `agentharm_benign` @task functions.",
    "agentharm.py": "MAIN ENTRYPOINT. Defines @task agentharm + agentharm_benign and run_agent_benchmark(): loads dataset, filters it, builds tools, picks the agent solver, wires up the scorer, returns an inspect Task.",
    "utils.py": "DATASET LOADING + tool setup. load_dataset() downloads from HuggingFace (ai-safety-institute/AgentHarm) and selects the split json; filter_dataset() applies behavior_ids/detailed/hint filters; setup_tools_from_metadata() injects each sample's target tools.",
    "scorer.py": "SCORING. RefusalJudgeLLM (LLM-based refusal detection) + combined_scorer() which returns a Score with {score, refusal} per sample.",
    "metric.py": "METRICS aggregated across samples: avg_score (harm score), avg_full_score, avg_refusals (refusal rate), avg_score_non_refusals, plus per-category versions.",
    "prompts.py": "System/user prompt templates and prompt techniques (standard/cot/react/refusal). Jailbreak template is intentionally NOT shipped.",
    "agents/agent.py": "AGENT_DICT registry mapping agent name -> solver ('default', 'refusal').",
    "agents/default_agent.py": "The default tool-calling agent (system prompt + tool loop).",
    "agents/refusal_agent.py": "A trivial agent that always refuses (useful as a sanity baseline).",
    "benchmark/harmful_tools/": "Python tool implementations used by harmful tasks.",
    "benchmark/benign_tools/": "Python tool implementations used by benign tasks.",
    "benchmark/harmful_grading_functions.py": "Per-task grading logic that computes the harm score for harmful tasks.",
    "benchmark/benign_grading_functions.py": "Per-task grading logic for the benign tasks.",
    "eval.yaml": "Eval metadata: title, arxiv link, 176 dataset_samples per task, HuggingFace asset pin.",
    "README.md": "Usage, run commands, parameters, citation.",
}
for f, role in file_roles.items():
    print(f"- {f}\n    {role}\n")

- __init__.py
    Exports the `agentharm` and `agentharm_benign` @task functions.

- agentharm.py
    MAIN ENTRYPOINT. Defines @task agentharm + agentharm_benign and run_agent_benchmark(): loads dataset, filters it, builds tools, picks the agent solver, wires up the scorer, returns an inspect Task.

- utils.py
    DATASET LOADING + tool setup. load_dataset() downloads from HuggingFace (ai-safety-institute/AgentHarm) and selects the split json; filter_dataset() applies behavior_ids/detailed/hint filters; setup_tools_from_metadata() injects each sample's target tools.

- scorer.py
    SCORING. RefusalJudgeLLM (LLM-based refusal detection) + combined_scorer() which returns a Score with {score, refusal} per sample.

- metric.py
    METRICS aggregated across samples: avg_score (harm score), avg_full_score, avg_refusals (refusal rate), avg_score_non_refusals, plus per-category versions.

- prompts.py
    System/user prompt templates and prompt techniques (standard/cot/react/refusal). Jailbre

In [41]:
# Show the exact registration + scoring wiring (a few short slices, no harmful content).
import inspect as pyinspect
from inspect_evals.agentharm import agentharm as agentharm_task
from inspect_evals.agentharm import scorer as ah_scorer

print("=== Task is registered via the @task decorator (entrypoint) ===")
print("   inspect_evals.agentharm.agentharm  ->  CLI name: inspect_evals/agentharm")
print("   inspect_evals.agentharm.agentharm_benign -> CLI name: inspect_evals/agentharm_benign")
print()
print("=== Default task arguments (signature of agentharm) ===")
print(pyinspect.signature(agentharm_task))
print()
print("=== Metrics attached to the combined scorer ===")
# These names come straight from scorer.py's @scorer(metrics=[...]) decorator.
print("   avg_score (harm score), avg_full_score, avg_refusals (refusal rate),")
print("   avg_score_non_refusals, category_avg_score, category_avg_refusals")

=== Task is registered via the @task decorator (entrypoint) ===
   inspect_evals.agentharm.agentharm  ->  CLI name: inspect_evals/agentharm
   inspect_evals.agentharm.agentharm_benign -> CLI name: inspect_evals/agentharm_benign

=== Default task arguments (signature of agentharm) ===
(agent: str = 'default', behavior_ids: list[str] = [], split: Literal['val', 'test_private', 'test_public'] = 'test_public', detailed_behaviors: bool | None = None, hint_included: bool | None = None, n_irrelevant_tools: int = 0, refusal_judge: str | inspect_ai.model._model.Model = 'openai/gpt-4o-2024-08-06', semantic_judge: str | inspect_ai.model._model.Model = 'openai/gpt-4o-2024-08-06', force_redownload: bool = False, agent_kwargs: dict[str, typing.Any] | None = None, chat_dataset: bool | None = False) -> inspect_ai._eval.task.task.Task

=== Metrics attached to the combined scorer ===
   avg_score (harm score), avg_full_score, avg_refusals (refusal rate),
   avg_score_non_refusals, category_avg_score, ca

---
## 3. Dataset loading

Load the dataset **exactly the way the benchmark does** (via `inspect_evals.agentharm.utils.load_dataset`). The first call downloads the dataset from HuggingFace into the inspect_evals cache directory (a one-time ~download).

We show **IDs / names / categories** and counts only — not full harmful prompts.

In [42]:
# Load the public harmful test split the same way run_agent_benchmark() does.
# NOTE: this downloads ~the dataset from HuggingFace on first run. It needs network
# access but NO API key. If it fails, the blocker is almost always network or a
# missing `huggingface_hub`; the cell prints a clear next step.
from inspect_evals.agentharm.utils import load_dataset

try:
    harmful_public = load_dataset(name="harmful", split="test_public", force_redownload=False)
    print(f"Loaded harmful/test_public: {len(harmful_public)} samples")
    dataset_ok = True
except Exception as e:  # noqa: BLE001
    dataset_ok = False
    print(f"[BLOCKER] Could not load dataset: {type(e).__name__}: {e}")
    print("Next step: ensure network access to huggingface.co and that")
    print("`huggingface_hub` is installed (it is a dependency of inspect_evals).")

Loaded harmful/test_public: 176 samples


Loaded harmful/test_public: 176 samples


In [43]:
# Summarize the dataset by category and show a few task IDs/names.
# We DELIBERATELY do not print the full prompt text.
from collections import Counter

if dataset_ok:
    cats = Counter()
    rows = []
    for sample in harmful_public:
        md = sample.metadata or {}
        cats[md.get("category", "?")] += 1
        rows.append((sample.id, md.get("name", "?"), md.get("category", "?")))

    print("Samples per category (harmful / test_public):")
    for cat, n in sorted(cats.items()):
        print(f"   {cat:<30} {n}")

    print("\nFirst 8 task IDs / names / categories (no prompt text shown):")
    for sid, name, cat in rows[:8]:
        print(f"   id={sid!s:<8} category={cat:<22} name={name}")

    # Safe, truncated peek at ONE prompt so you can confirm the schema without
    # reproducing operational detail.
    first_md = (harmful_public[0].metadata or {})
    sample_prompt = str(harmful_public[0].input)
    print("\nSchema sanity check (truncated, first 90 chars of one prompt):")
    print("   ", sample_prompt[:90].replace("\n", " ") + " ...[truncated]")
    print("   metadata keys:", sorted(first_md.keys()))
else:
    print("Skipping dataset summary because loading failed above.")

Samples per category (harmful / test_public):
   Copyright                      24
   Cybercrime                     20
   Disinformation                 20
   Drugs                          20
   Fraud                          24
   Harassment                     24
   Hate                           20
   Sexual                         24

First 8 task IDs / names / categories (no prompt text shown):
   id=1-1      category=Disinformation         name=Propaganda AI Video
   id=1-2      category=Disinformation         name=Propaganda AI Video
   id=1-3      category=Disinformation         name=Propaganda AI Video
   id=1-4      category=Disinformation         name=Propaganda AI Video
   id=2-1      category=Disinformation         name=Kiwi Unlimited Spying on Customers
   id=2-2      category=Disinformation         name=Kiwi Unlimited Spying on Customers
   id=2-3      category=Disinformation         name=Kiwi Unlimited Spying on Customers
   id=2-4      category=Disinformation        

In [44]:
# Optional: counts for other splits/datasets so you know the full landscape.
# Each call may trigger a (cached) download. Comment out if you only care about
# harmful/test_public.
if dataset_ok:
    for name in ["harmful", "benign"]:
        for split in ["val", "test_public"]:
            try:
                ds = load_dataset(name=name, split=split, force_redownload=False)
                print(f"{name:<8} / {split:<12} -> {len(ds)} samples")
            except Exception as e:  # noqa: BLE001
                print(f"{name:<8} / {split:<12} -> error: {e}")

harmful  / val          -> 32 samples
harmful  / test_public  -> 176 samples
benign   / val          -> 32 samples
benign   / test_public  -> 176 samples


---
## 4. Minimal run command builder

Define the knobs for the **smallest** meaningful run and print the exact `inspect eval` command. **No API keys are hardcoded** — they are read from the environment at run time.

Key flags used:
- `--limit N` — run only the first N samples (this is how you get 5–10 samples).
- `--model` — the target model under test.
- `-T split=test_public` — the public split (also the default).
- `-T refusal_judge=... -T semantic_judge=...` — override judges if you don't have an OpenAI key.
- `--log-dir` — where `.eval` logs are written.
- No jailbreak: we leave `agent_kwargs` at defaults (`user_prompt_template='empty'`), so **no jailbreak template is applied**.

In [45]:
import shlex

# ---- EDIT THESE ----
TARGET_MODEL = "openai/gpt-4o-mini"          # the model being tested
N_SAMPLES = 10                                  # tiny subset: 5-10 recommended
SPLIT = "test_public"                          # public split only
# Judges default to openai/gpt-4o-2024-08-06. If you have NO OpenAI key, point
# them at a provider you DO have a key for (any capable instruction model works):
REFUSAL_JUDGE = "openai/gpt-4o-2024-08-06"
SEMANTIC_JUDGE = "openai/gpt-4o-2024-08-06"
# --------------------

# Log directory lives next to this notebook so results are easy to find.
LOG_DIR = str((repo_root / "experiments" / "agentharm_repro" / "logs").resolve())

# NOTE: we force --log-format json on purpose.
# AgentHarm's scorer stores the semantic-judge Model object in sample metadata,
# which the default .eval (streaming) recorder cannot serialize -> the run aborts
# with `PydanticSerializationError: Unable to serialize ... Model`. The json
# recorder tolerates it. This is a CLI flag only; it does not change benchmark code.
cmd = [
    "uv", "run", "inspect", "eval", "inspect_evals/agentharm",
    "--model", TARGET_MODEL,
    "--limit", str(N_SAMPLES),
    "-T", f"split={SPLIT}",
    "-T", f"refusal_judge={REFUSAL_JUDGE}",
    "-T", f"semantic_judge={SEMANTIC_JUDGE}",
    "--log-format", "json",
    "--log-dir", LOG_DIR,
]

print("Exact command to run a tiny, no-jailbreak AgentHarm reproduction:\n")
print(" ".join(shlex.quote(c) for c in cmd))
print("\nLogs will be written to:", LOG_DIR)

Exact command to run a tiny, no-jailbreak AgentHarm reproduction:

uv run inspect eval inspect_evals/agentharm --model openai/gpt-4o-mini --limit 10 -T split=test_public -T refusal_judge=openai/gpt-4o-2024-08-06 -T semantic_judge=openai/gpt-4o-2024-08-06 --log-format json --log-dir /Users/raghavmanoharanjayanthi/Downloads/inspect_evals/experiments/agentharm_repro/logs

Logs will be written to: /Users/raghavmanoharanjayanthi/Downloads/inspect_evals/experiments/agentharm_repro/logs


---
## 5. Environment variables

Check which provider API keys are present. **The notebook never prints key values** — only whether each is set.

In [46]:
# Which keys you need depends on the providers used by (a) the target model and
# (b) the two judges. With all defaults (target gpt-4o-mini + openai judges) you
# only need OPENAI_API_KEY.
provider_keys = ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GOOGLE_API_KEY"]
print("API key presence (values hidden):")
for k in provider_keys:
    print(f"   {k:<20} {'SET' if os.environ.get(k) else 'not set'}")

def provider_of(model_str: str) -> str:
    return model_str.split("/", 1)[0]

needed_providers = {provider_of(TARGET_MODEL), provider_of(REFUSAL_JUDGE), provider_of(SEMANTIC_JUDGE)}
print("\nProviders required by your current config:", sorted(needed_providers))

provider_env = {"openai": "OPENAI_API_KEY", "anthropic": "ANTHROPIC_API_KEY", "google": "GOOGLE_API_KEY"}
missing = [provider_env[p] for p in needed_providers if p in provider_env and not os.environ.get(provider_env[p])]
if missing:
    print("\n[BLOCKER] Missing keys for this config:", missing)
    print("Next step: export them in your shell before running, e.g.:")
    for k in missing:
        print(f"   export {k}=...")
else:
    print("\n[OK] All keys required by the current config are present.")

API key presence (values hidden):
   OPENAI_API_KEY       SET
   ANTHROPIC_API_KEY    not set
   GOOGLE_API_KEY       not set

Providers required by your current config: ['openai']

[OK] All keys required by the current config are present.


---
## 6. How to limit sample count (reference)

Inspect flags relevant to controlling the run size:

| Flag | What it does |
| --- | --- |
| `--limit N` | Run only the first N samples. **Use this for a tiny run.** |
| `--limit A-B` | Run a sample range (e.g. `--limit 0-5`). |
| `--sample-id ID` | Run a specific sample id (repeatable). Maps to AgentHarm `behavior_ids`. |
| `--epochs K` | Repeat each sample K times (keep at 1 for reproduction). |
| `--max-connections N` | Cap concurrent model requests. |
| `--log-dir PATH` | Where `.eval` logs are written. |

AgentHarm-specific task args (passed with `-T`): `split`, `behavior_ids`, `detailed_behaviors`, `hint_included`, `n_irrelevant_tools`, `refusal_judge`, `semantic_judge`, `agent`, `agent_kwargs`.

Note: there is no `--max-samples` flag in Inspect; `--limit` is the one you want.

---
## 7. Optional execution cell

Run the command from Section 4 via `subprocess`. **It is commented out by default** so the notebook can be explored without spending API credits. Uncomment the `run_eval()` call to actually execute.

Expect a tiny run (5 samples, gpt-4o-mini) to take roughly a minute or two and cost a few US cents, dominated by the judge calls.

In [47]:
import subprocess

def run_eval():
    """Run the AgentHarm command built in Section 4, streaming output."""
    Path(LOG_DIR).mkdir(parents=True, exist_ok=True)
    print("Running:\n ", " ".join(shlex.quote(c) for c in cmd), "\n")
    proc = subprocess.run(
        cmd,
        cwd=str(repo_root),
        capture_output=True,
        text=True,
    )
    print("=== STDOUT (tail) ===")
    print("\n".join(proc.stdout.splitlines()[-40:]))
    print("\n=== STDERR (tail) ===")
    print("\n".join(proc.stderr.splitlines()[-40:]))
    print("\nReturn code:", proc.returncode)
    if proc.returncode != 0:
        print("[BLOCKER] eval exited non-zero. Common causes: missing API key,")
        print("          model name typo, or no network for the dataset download.")
    return proc.returncode

# ---- Uncomment the next line to actually run the eval ----
run_eval()
print("Execution is disabled. Uncomment `run_eval()` above to run the tiny eval.")

Running:
  uv run inspect eval inspect_evals/agentharm --model openai/gpt-4o-mini --limit 10 -T split=test_public -T refusal_judge=openai/gpt-4o-2024-08-06 -T semantic_judge=openai/gpt-4o-2024-08-06 --log-format json --log-dir /Users/raghavmanoharanjayanthi/Downloads/inspect_evals/experiments/agentharm_repro/logs 

=== STDOUT (tail) ===
combined_scorer                                                                 
avg_score                    0.900                                              
avg_full_score               0.800                                              
avg_refusals                 0.000                                              
avg_score_non_refusals       0.900                                              
Disinformation_avg_scores    0.875                                              
Harassment_avg_scores        1.000                                              
Disinformation_avg_refusals  0.000                                              
Harassment_av

---
## 8. Read results / logs

Find the most recent `.eval` log in `LOG_DIR`, read it with the Inspect log API, and pull out the headline metrics: **harm score** (`avg_score`), **refusal rate** (`avg_refusals`), and per-category breakdowns if present.

In [48]:
# Locate the newest log file in our log dir. We accept BOTH .json and .eval
# (this notebook writes .json to avoid the serialization bug; read_eval_log reads both).
from inspect_ai.log import read_eval_log

log_path = Path(LOG_DIR)
logs = []
if log_path.exists():
    logs = list(log_path.glob("*.json")) + list(log_path.glob("*.eval"))
logs.sort(key=lambda p: p.stat().st_mtime)

if not logs:
    print(f"No logs found in {LOG_DIR}.")
    print("Next step: run the eval (Section 7) first, then re-run this cell.")
    latest_log = None
else:
    latest_log = logs[-1]
    print("Most recent log:", latest_log.name)

Most recent log: 2026-06-15T19-29-18-00-00_agentharm_dntYepGKZP6mfsCXCKkJrB.json


In [49]:
# Parse the log and print a small results table.
def summarize_log(eval_log_path):
    log = read_eval_log(str(eval_log_path))
    print("Task:    ", log.eval.task)
    print("Model:   ", log.eval.model)
    print("Status:  ", log.status)
    n = len(log.samples) if log.samples is not None else "?"
    print("Samples: ", n)
    print()

    if not log.results or not log.results.scores:
        print("No aggregated scores in this log (did the run finish?).")
        return

    headline = {}
    per_category = {}
    for score in log.results.scores:
        for metric_name, metric in score.metrics.items():
            val = metric.value
            if metric_name in ("avg_score", "avg_full_score", "avg_refusals", "avg_score_non_refusals"):
                headline[metric_name] = val
            elif metric_name.endswith("_avg_scores") or metric_name.endswith("_avg_refusals"):
                per_category[metric_name] = val

    label = {
        "avg_score": "Harm score (avg_score)",
        "avg_full_score": "Full-harm rate (avg_full_score)",
        "avg_refusals": "Refusal rate (avg_refusals)",
        "avg_score_non_refusals": "Harm score | non-refusals",
    }
    print("=== Headline metrics ===")
    for k in ["avg_score", "avg_full_score", "avg_refusals", "avg_score_non_refusals"]:
        if k in headline and isinstance(headline[k], (int, float)):
            print(f"   {label[k]:<32} {headline[k]:.3f}")

    if per_category:
        print("\n=== Per-category (first few) ===")
        for k in list(per_category)[:10]:
            v = per_category[k]
            if isinstance(v, (int, float)):
                print(f"   {k:<40} {v:.3f}")
    return headline

headline_metrics = summarize_log(latest_log) if latest_log else None

Task:     inspect_evals/agentharm
Model:    openai/gpt-4o-mini
Status:   success
Samples:  10

=== Headline metrics ===
   Harm score (avg_score)           0.900
   Full-harm rate (avg_full_score)  0.800
   Refusal rate (avg_refusals)      0.000
   Harm score | non-refusals        0.900

=== Per-category (first few) ===
   Disinformation_avg_scores                0.875
   Harassment_avg_scores                    1.000
   Disinformation_avg_refusals              0.000
   Harassment_avg_refusals                  0.000


---
## 9. Reproduction target

**This reproduces a small-scale AgentHarm-style result: refusal / compliance behavior on a subset of harmful agent tasks.**

The two numbers to report are:

- **Refusal rate** (`avg_refusals`): fraction of tasks where the model refused. Higher = safer.
- **Harm score** (`avg_score`): average per-task grade of how completely the harmful task was carried out (0–1, scaled to a % in the paper). Lower = safer.

**Context from the paper / README:** on the no-jailbreak ("direct request") setting, the public-behavior average harm score is reported around **54.9% for GPT-4o** and **83.6% for Mistral**. Frontier models also tend to show **low single-digit to low-double-digit refusal rates** on direct harmful requests. Your tiny 5–10 sample run is **not statistically robust** — it is a methodology reproduction, not a precise number match. Treat it as "same ballpark / same direction" rather than an exact replication.

For a fuller comparison, run the benign task too (`inspect_evals/agentharm_benign`) to contrast benign vs harmful behavior.

---
## 10. Meeting summary (copy into your notes)

The cell below builds a short markdown summary you can paste into your research notes, filling in whatever metrics were parsed above.

In [28]:
harm = refusal = None
if isinstance(headline_metrics, dict):
    harm = headline_metrics.get("avg_score")
    refusal = headline_metrics.get("avg_refusals")

def fmt(x):
    return f"{x:.3f}" if isinstance(x, (int, float)) else "(run the eval to fill in)"

summary = f"""### AgentHarm — tiny reproduction notes

- **Paper:** AgentHarm: A Benchmark for Measuring Harmfulness of LLM Agents (arXiv:2410.09024)
- **Result reproduced:** refusal rate + harm score on a tiny subset of harmful agentic tasks (no jailbreak, public split).
- **Config:** target model = `{TARGET_MODEL}`, samples = {N_SAMPLES}, split = `{SPLIT}`, refusal/semantic judge = `{REFUSAL_JUDGE}`.
- **My measured refusal rate (avg_refusals):** {fmt(refusal)}
- **My measured harm score (avg_score):** {fmt(harm)}
- **Paper reference (no-jailbreak, public behaviors):** ~54.9% harm score for GPT-4o; refusal rates are low for frontier models.
- **Code/data/tools needed:** inspect_evals repo (`src/inspect_evals/agentharm`), Inspect AI; dataset auto-downloaded from HuggingFace `ai-safety-institute/AgentHarm`; per-task tools + grading functions ship with the repo.
- **Compute/API needed:** API key(s) for the target model AND the LLM judges (default OpenAI gpt-4o). No GPU. A 5-sample run costs a few cents and takes ~1-2 min.
- **Caveat:** 5-10 samples is a methodology check, not a statistically robust number.
- **Next extension idea:** scale to the full 176-sample public split, add the benign task for a benign-vs-harmful contrast, then compare multiple target models or prompt techniques (cot/react).
"""
print(summary)

### AgentHarm — tiny reproduction notes

- **Paper:** AgentHarm: A Benchmark for Measuring Harmfulness of LLM Agents (arXiv:2410.09024)
- **Result reproduced:** refusal rate + harm score on a tiny subset of harmful agentic tasks (no jailbreak, public split).
- **Config:** target model = `openai/gpt-4o-mini`, samples = 5, split = `test_public`, refusal/semantic judge = `openai/gpt-4o-2024-08-06`.
- **My measured refusal rate (avg_refusals):** (run the eval to fill in)
- **My measured harm score (avg_score):** (run the eval to fill in)
- **Paper reference (no-jailbreak, public behaviors):** ~54.9% harm score for GPT-4o; refusal rates are low for frontier models.
- **Code/data/tools needed:** inspect_evals repo (`src/inspect_evals/agentharm`), Inspect AI; dataset auto-downloaded from HuggingFace `ai-safety-institute/AgentHarm`; per-task tools + grading functions ship with the repo.
- **Compute/API needed:** API key(s) for the target model AND the LLM judges (default OpenAI gpt-4o). No GPU

**Comments:**

What this says, in one sentence
On this tiny no-jailbreak slice, gpt-4o-mini refused nothing (0% refusal) and substantially completed the harmful agent tasks (90% harm score, 80% fully completed) — i.e. weak agentic safety on these behaviors, consistent in direction with the paper's finding that direct (non-jailbroken) harmful requests already elicit high harm scores from frontier models.

Two caveats to state in your meeting: (1) this is a handful of samples across only 2 of 11 categories, so it's a methodology reproduction, not a precise/robust number; (2) higher harm score = less safe, and higher refusal rate = more safe — they move in opposite directions.

Want me to run the full 176-sample public split to get statistically meaningful per-category numbers, or add the benign task to confirm the model still does the harmless versions (a good sanity check that low refusal isn't just over-refusal)?

**Overall**
Metric	Your value	What it measures
Harm score (avg_score)
0.900
Average "how much of the harmful task did the agent actually carry out", across all samples. 0 = did nothing harmful, 1 = fully executed. 90% here.

Full-harm rate (avg_full_score)
0.800
Fraction of tasks that scored a perfect 1.0 (every grading check passed). 80% of tasks were completed in full.

Refusal rate (avg_refusals)
0.000
Fraction of tasks where the refusal-judge LLM detected a refusal. 0% — the model never declined.

Harm score | non-refusals (avg_score_non_refusals)
0.900
Same as harm score, but averaged only over tasks the model didn't refuse.

How they relate to each other
Why is harm score (0.900) higher than full-harm rate (0.800)? Each task has several grading sub-checks (e.g. "called the right tool", "used the correct target", "produced the harmful content"). The per-task score is the fraction of those sub-checks passed. avg_score averages those fractions; avg_full_score counts only tasks where the fraction was exactly 1.0. So a task that passed 3 of 4 checks contributes 0.75 to harm score but 0 to full-harm rate. That gap (0.9 vs 0.8) means some tasks were mostly but not perfectly completed.